# Fine-tuning and Converting LLMs with Unsloth and llama.cpp

This notebook demonstrates how to:
1. Fine-tune a language model using Unsloth with LoRA adaptation.
2. Convert the fine-tuned model to GGUF format for use with llama.cpp.
3. Run the model locally using llama.cpp.

## Table of Contents
1. [Setup](#Setup)
2. [Fine-tuning with Unsloth](#Fine-tuning-with-Unsloth)
3. [Conversion to GGUF](#Conversion-to-GGUF)
4. [Running with llama.cpp](#Running-with-llama.cpp)

---


## 1. Setup

First, let's install the necessary packages. We'll need:
- unsloth
- torch
- transformers
- datasets
- trl
- peft

Note: If you are running this notebook in an environment where these packages are not installed, you can install them using pip.

We'll also need llama.cpp for the conversion and inference. We assume you have cloned the llama.cpp repository.
If not, you can clone it with:
```bash
git clone https://github.com/ggerganov/llama.cpp
```

And then install the required Python packages for the conversion script:
```bash
pip install -r llama.cpp/requirements.txt
```

Let's check if we have the necessary packages and install if missing.

We'll also set an environment variable for the llama.cpp directory to make the conversion script work.


In [ ]:
# Install required packages
!pip install unsloth torch transformers datasets trl peft -q

# Set the environment variable for llama.cpp directory
import os
# Adjust this path to where you cloned llama.cpp
os.environ['LLAMA_CPP_DIR'] = os.path.join(os.getcwd(), 'llama.cpp')
print(f"LLAMA_CPP_DIR set to: {os.environ['LLAMA_CPP_DIR']}")

# Check if the directory exists
if not os.path.exists(os.environ['LLAMA_CPP_DIR']):
    print("Warning: LLAMA_CPP_DIR does not exist. Please clone llama.cpp first.")
    print("You can do so by running: git clone https://github.com/ggerganov/llama.cpp")


## 2. Fine-tuning with Unsloth

We'll fine-tune a small model (TinyLlama) on a custom dataset using LoRA adaptation.
This approach is memory-efficient and can run on consumer-grade GPUs.

### Step 2.1: Load the model and tokenizer

We'll use the `unsloth/TinyLlama-V1.1-Chat` model as our base model.
We'll load it in 4-bit quantization to reduce memory usage.


In [ ]:
import torch
from unsloth import FastLanguageModel

model_name = "unsloth/TinyLlama-V1.1-Chat"
max_seq_length = 2048
dtype = torch.float16  # Use float16 for better performance
load_in_4bit = True    # Use 4-bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # if using a gated model
)

print(f"Model loaded: {model_name}")
print(f"Model dtype: {dtype}")
print(f"Load in 4-bit: {load_in_4bit}")


### Step 2.2: Configure LoRA

We'll configure LoRA (Low-Rank Adaptation) to fine-tune the model efficiently.
We'll target the query, key, value, and output projection matrices, as well as the gate, up, and down projection matrices in the MLP layers.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
    "                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,  # Supports any, but = 0 is optimized
    bias = "none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for very long context
    random_state = 3407,
    max_seq_length = max_seq_length,
)

print("LoRA configuration applied.")
print(f"Model trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


### Step 2.3: Prepare the dataset

We'll create a small custom dataset for demonstration purposes.
In practice, you would replace this with your own dataset.

The dataset will consist of instruction-response pairs in Turkish.
We'll format them in a way that is suitable for chat-style fine-tuning.


In [ ]:
from datasets import Dataset

# Small dataset for demonstration
data = [
    {"instruction": "Merhaba, nasılsın?", "response": "İyiyim, teşekkür ederim! Sen nasılsın?"},
    {"instruction": "Türkiye'nin başkenti neresidir?", "response": "Türkiye'nin başkenti Ankara'dır."},
    {"instruction": "Python'da bir listeyi nasıl sıralarsın?", "response": "Python'da bir listeyi sorted() fonksiyonu veya list.sort() metodu kullanarak sıralayabilirsin."},
    {"instruction": "E=mc² formülüsünü açıklayabilir misin?", "response": "E=mc² formülü, Einstein'ın görecelilik teorisinden ku خورشur ve enerji (E), kütle (m) ve ışık hızının karesi (c²) arasındaki ilişkiyi gösterir."},
    {"instruction": "İstanbul'un ünlü müzeleri nelerdir?", "response": "İstanbul'un ünlü müzeleri arasında Topkapı Sarayı, Hagia Sophia, İstanbul Arkeoloji Müzesi, Türk ve İslam Eserleri Müzesi ve Pera Müzesi bulunur."},
]

# Convert to Hugging Face Dataset
def format_instruction(example):
    return {
        "text": f"### Kullanıcı:\n{example['instruction']}\n\n### Asistan:\n{example['response']}"
    }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_instruction, remove_columns=["instruction", "response"])

print(f"Dataset size: {len(dataset)}")
print("Example entry:")
print(dataset[0]['text'])


### Step 2.4: Configure the trainer

We'll use the SFTTrainer from the trl library for supervised fine-tuning.
We'll set up the training arguments for a quick demo (1 epoch).
For real training, you would increase the number of epochs and adjust the batch size and learning rate.


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,  # Can set to True for faster training
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,  # Set to 1 for demo; increase for real training
        learning_rate = 2e-4,
        fp16 = not load_in_4bit,  # Use FP16 if not 4-bit
        bf16 = False,  # Set to True if using Ampere or newer GPU
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",  # Use for Weights and Biases
    ),
)

print("Trainer configured.")


### Step 2.5: Train the model

Let's start the training process. This will take a few minutes depending on your hardware.

Note: Since we are using a very small dataset and only 1 epoch, this is just for demonstration.
For real-world tasks, you would use a larger dataset and train for more epochs.


In [ ]:
print("Starting training...")
trainer.train()
print("Training completed!")


### Step 2.6: Save the fine-tuned model

After training, we'll save the model and tokenizer locally.
We'll also show how to push to the Hugging Face Hub (optional).


In [ ]:
print("Saving model...")
model.save_pretrained("fine_tuned_model")  # Local saving
tokenizer.save_pretrained("fine_tuned_model")

# Optional: Save to Hugging Face Hub
# model.push_to_hub("your_username/fine_tuned_model", token = "hf_...")
# tokenizer.push_to_hub("your_username/fine_tuned_model", token = "hf_...")

print("Model saved to 'fine_tuned_model' directory.")


## 3. Conversion to GGUF

Now we'll convert the fine-tuned model to GGUF format for use with llama.cpp.
We'll use the conversion script from the llama.cpp repository.

First, let's make sure we have the llama.cpp repository cloned and the conversion script available.
We'll use the `convert_to_gguf.py` script we created earlier.


In [ ]:
# First, let's check if we have the conversion script
import os
from pathlib import Path

# We'll use the convert_to_gguf.py script we created
convert_script = Path(__file__).parent / "convert_to_gguf.py"
if convert_script.exists():
    print(f"Conversion script found at: {convert_script}")
else:
    print(f"Conversion script not found at: {convert_script}")
    print("Please make sure you have created the convert_to_gguf.py script.")

# We'll also check if the fine-tuned model exists
model_path = Path(__file__).parent / "fine_tuned_model"
if model_path.exists():
    print(f"Fine-tuned model found at: {model_path}")
else:
    print(f"Fine-tuned model not found at: {model_path}")
    print("Please make sure you have completed the fine-tuning step above.")


Now, let's run the conversion. We'll convert the model to GGUF format with f16 precision (float16).
You can also choose other quantization types like q4_0, q5_1, etc., for smaller file sizes.

We'll use the `convert_to_gguf.py` script we created earlier.

Note: The conversion process requires the llama.cpp conversion script to be available.
We set the LLAMA_CPP_DIR environment variable earlier to point to the llama.cpp repository.


In [ ]:
# Run the conversion
!python "convert_to_gguf.py" "fine_tuned_model" --outtype f16


After the conversion completes, you should see a GGUF file in the `fine_tuned_model` directory.
The file will be named something like `model.f16.gguf`.

Let's check what files are in the directory now.


In [ ]:
!ls -la "fine_tuned_model"


## 4. Running with llama.cpp

Now that we have the GGUF model, we can run it locally using llama.cpp.

There are two main ways to run the model:
1. Using the `llama-cli` command line interface.
2. Using the `llama-server` to start a local server that offers an API similar to OpenAI's.

We'll demonstrate both.

First, let's make sure we have built the llama.cpp binaries.
If you haven't built them yet, you can do so with:
```bash
cd llama.cpp
make clean
make -j$(nproc)
```

Assuming you have built the binaries, let's proceed.


### 4.1: Using llama-cli

We'll run an interactive session where we can chat with the model.

Note: The exact command may vary depending on your system and the binaries you built.
We'll assume the binaries are in the `llama.cpp` directory.


In [ ]:
# Example command for llama-cli (adjust paths as needed)
# We'll use the GGUF model we just converted

gguf_model_path = "fine_tuned_model/model.f16.gguf"

# Check if the model exists
import os
if os.path.exists(gguf_model_path):
    print(f"GGUF model found at: {gguf_model_path}")
    print("\nTo run the model with llama-cli, use a command like:")
    print(f"  llama.cpp/llama-cli -m {gguf_model_path} --color -c 2048 --temp 0.7 --repeat_penalty 1.1\n")
    print("Then you can start chatting with the model. Type your prompts and press Enter.")
    print("To exit, type '/exit' or press Ctrl+C.\n")
else:
    print(f"GGUF model not found at: {gguf_model_path}")
    print("Please make sure the conversion step completed successfully.")


### 4.2: Using llama-server

We can also start a local server that provides an API endpoint similar to OpenAI's.
This allows us to interact with the model using HTTP requests.

We'll start the server in the background and then send a simple request to it.

Note: Make sure to stop the server when you're done to free up resources.


In [ ]:
# Example command for llama-server
# We'll start the server on port 8080

gguf_model_path = "fine_tuned_model/model.f16.gguf"

if os.path.exists(gguf_model_path):
    print(f"GGUF model found at: {gguf_model_path}")
    print("\nTo start the server, use a command like:")
    print(f"  llama.cpp/llama-server -m {gguf_model_path} --port 8080\n")
    print("Then you can send requests to http://localhost:8080/v1/completions")
    print("Example request (in another terminal or using curl):")
    print('''  curl -X POST http://localhost:8080/v1/competions \
    -H "Content-Type: application/json" \
    -d '{"model": "local-model", "prompt": "Merhaba, nasılsın?", "max_tokens": 50, "temperature": 0.7}'")
    print("\nTo stop the server, press Ctrl+C in the terminal where it's running.\n")
else:
    print(f"GGUF model not found at: {gguf_model_path}")
    print("Please make sure the conversion step completed successfully.")


## Summary

In this notebook, we have:
1. Fine-tuned a language model (TinyLlama) using Unsloth with LoRA adaptation.
2. Converted the fine-tuned model to GGUF format for use with llama.cpp.
3. Shown how to run the model locally using llama-cli and llama-server.

## Next Steps

- Experiment with different datasets and fine-tuning parameters.
- Try different quantization types (q4_0, q5_1, etc.) when converting to GGUF to balance size and quality.
- Explore the llama.cpp documentation for more advanced usage options.
- Consider using the model in applications via the llama-server API.

---

*Note: This notebook is for demonstration purposes. For production use, you would want to use a larger dataset, train for more epochs, and carefully evaluate the model's performance.*
